# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [ ]:

# Task 1 — Fossil fuel treemap

# Filter fossil sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe colour palette
fossil_colours = {
    'Coal': '#D55E00',        # Vermillion
    'Oil': '#E69F00',         # Orange
    'Natural Gas': '#0072B2'  # Blue
}

# Calculate leading region and country by fossil TWh
country_fossil = (
    df_fossil.groupby('Country')['TWh']
    .sum()
    .sort_values(ascending=False)
)

region_fossil = (
    df_fossil.groupby('Region')['TWh']
    .sum()
    .sort_values(ascending=False)
)

top_country = country_fossil.index[0]
top_region = region_fossil.index[0]

# Create treemap
fig1 = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colours,
    title=(
        f"Fossil Fuel Dependency: {top_region} leads by region "
        f"and {top_country} by country"
    )
)

# Grey out parent nodes; colour only the source-level nodes
labels = list(fig1.data[0].labels)
fig1.data[0].marker.colors = [
    fossil_colours.get(label, '#BDBDBD')
    for label in labels
]

# Display TWh values, not percentages
fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Energy: %{value:,.1f} TWh'
        '<extra></extra>'
    )
)

fig1.update_layout(
    margin=dict(t=80, l=10, r=10, b=10)
)

fig1.show()

print("Highest fossil fuel consumption by country:", top_country)
print("Highest fossil fuel consumption by region:", top_region)


Highest fossil fuel consumption by country: Saudi Arabia
Highest fossil fuel consumption by region: Asia


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:

# Task 2 — Low-carbon energy

# Filter low-carbon sources
df_lowcarbon = df.loc[
    df['Source_Type'] == 'Low-carbon'
].copy()

# Aggregate TWh by country
lowcarbon_country = (
    df_lowcarbon.groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

# Identify leading country
top_lowcarbon = lowcarbon_country.iloc[0]['Country']
top_lowcarbon_twh = lowcarbon_country.iloc[0]['TWh']

# Add dummy root for treemap
lowcarbon_country['All'] = 'Low-carbon'

# 1. Treemap
fig2 = px.treemap(
    lowcarbon_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon Energy Production by Country'
)

fig2.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Energy: %{value:,.1f} TWh'
        '<extra></extra>'
    )
)

fig2.show()

# 2. Horizontal bar chart
fig3 = px.bar(
    lowcarbon_country.sort_values('TWh', ascending=True),
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#0072B2'],
    text='TWh',
    title=(
        f'Low-carbon Energy: {top_lowcarbon} Leads '
        f'with {top_lowcarbon_twh:,.1f} TWh'
    ),
    labels={
        'TWh': 'Low-carbon Energy (TWh)',
        'Country': 'Country'
    }
)

fig3.update_traces(
    texttemplate='%{text:,.1f} TWh',
    textposition='outside'
)

fig3.update_layout(
    showlegend=False,
    xaxis_title='Energy (TWh)',
    yaxis_title='Country'
)

fig3.show()

print(
    f"Leading country: {top_lowcarbon} "
    f"({top_lowcarbon_twh:,.1f} TWh)"
)


Leading country: France (7,593.0 TWh)


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

Because the countries are organized according to their overall production of low-carbon energy, the horizontal bar chart makes the ranking story easier to understand. Comparing values, determining which nation produces the most, and observing national disparities are all simple tasks. Although it is more challenging to compare precise figures and rankings, the treemap is superior for illustrating each nation's relative contribution to the total low-carbon energy mix.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:

# Task 3 — 100% stacked bar chart

# Reuse df_fossil from Task 1

# Aggregate fossil TWh by Region and Source
region_source = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

# Calculate each region's total fossil energy
region_source['Region_Total'] = (
    region_source.groupby('Region')['TWh'].transform('sum')
)

# Calculate percentage share
region_source['Share_pct'] = (
    region_source['TWh'] / region_source['Region_Total'] * 100
)

# Find coal share for each region
coal_shares = (
    region_source[region_source['Source'] == 'Coal']
    [['Region', 'Share_pct']]
    .sort_values('Share_pct', ascending=False)
)

top_coal_region = coal_shares.iloc[0]['Region']
top_coal_share = coal_shares.iloc[0]['Share_pct']

# Sort regions by coal share, highest first
region_order = coal_shares['Region'].tolist()

# Create labels for coal segments only
region_source['Coal_Label'] = np.where(
    region_source['Source'] == 'Coal',
    region_source['Share_pct'].map(lambda x: f'{x:.1f}%'),
    ''
)

# CVD-safe palette: same coal colour as Task 1
task3_colours = {
    'Coal': '#D55E00',
    'Oil': '#999999',
    'Natural Gas': '#CCCCCC'
}

# Create horizontal stacked bar chart
fig4 = px.bar(
    region_source,
    x='Share_pct',
    y='Region',
    color='Source',
    orientation='h',
    barmode='stack',
    text='Coal_Label',
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map=task3_colours,
    title=(
        f'{top_coal_region} is the Most Coal-Dependent Region '
        f'({top_coal_share:.1f}% of Fossil Energy)'
    ),
    labels={
        'Share_pct': 'Share of Fossil Energy (%)',
        'Region': 'Region',
        'Source': 'Energy Source'
    }
)

# Display percentage labels on coal segments only
fig4.update_traces(
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white'),
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Source: %{fullData.name}<br>'
        'Share: %{x:.1f}%'
        '<extra></extra>'
    )
)

# Ensure highest coal-dependent region appears at the top
fig4.update_layout(
    xaxis=dict(
        range=[0, 100],
        title='Share of Fossil Energy (%)',
        ticksuffix='%'
    ),
    yaxis=dict(
        title='Region',
        autorange='reversed'
    ),
    legend_title='Energy Source'
)

fig4.show()

print(
    f"Most coal-dependent region: {top_coal_region} "
    f"({top_coal_share:.1f}%)"
)


Most coal-dependent region: Africa (78.0%)


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*
The treemap in Task 1 illustrates which nations and energy sources account for the highest fossil fuel consumption in every region. It shows the hierarchical breakdown of fossil energy.

The Task 3 stacked bar chart answers which region has the highest proportion of coal in its fossil energy mix. Because all bars represent 100% of fossil energy, it simplifies the comparison of coal dependency across regions.